# ViTASA Enhanced — Pair Classification (Colab Free)

Run on Google Colab Free GPU (T4) with safe time limits.

**Formulation:** target-aspect pair classification (correct)
**Backbone:** ViSoBERT (pre-trained on social media)
**Metric:** macro F1 on 3 sentiment classes (loại 'none')

⏱️ **Thời gian an toàn cho Colab Free:**
- Baseline C1 đơn domain: ~15 phút ✅ SAFE
- Full ablation (4 configs × 3 domains × 10 epochs): ~2 tiếng ✅ SAFE
- Full ablation × 20 epochs: ~4 tiếng (có rủi ro timeout)

**Strategy:**
1. Chạy C1 (baseline) trước để xác nhận formulation đúng
2. Nếu OK → chạy C2-C4 từng domain 1 lần (tránh timeout)
3. Nếu timeout → giảm epochs hoặc chạy lại domain bị interrupt


In [ ]:
# 1. Clone dataset từ ViTASA repo gốc
!git clone https://github.com/kh4nh12/ViTASA.git ViTASA_repo 2>&1 | grep -E '(Cloning|clone|done)'
!echo "Dataset files:"
!ls -lh ViTASA_repo/*.jsonl

In [ ]:
# 2. Install dependencies
!pip install -q torch transformers scikit-learn seqeval underthesea
import torch
print(f"✅ PyTorch {torch.__version__}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

In [ ]:
# 3. Setup folder structure + copy dataset
import os
import shutil

os.makedirs('VITASA_Enhanced/baseline/data', exist_ok=True)
os.makedirs('VITASA_Enhanced/experiments/results_pair', exist_ok=True)

for domain in ['mobile', 'restaurant', 'hotel']:
    domain_dir = f'VITASA_Enhanced/baseline/data/{domain}'
    os.makedirs(domain_dir, exist_ok=True)
    src = f'ViTASA_repo/{domain}.jsonl'
    dst = f'{domain_dir}/{domain}.jsonl'
    if os.path.exists(src):
        shutil.copy(src, dst)
        lines = sum(1 for _ in open(dst))
        print(f"✅ {domain}: {lines} samples")

In [ ]:
# 4. Upload your code files
# Nếu có Google Drive: mount drive + copy từ drive
from google.colab import drive
drive.mount('/gdrive')
print("✅ Google Drive mounted")

In [ ]:
# Copy train_pair.py + modules từ Drive (adjust path nếu cần)
import shutil

# Giả sử bạn đã upload project vào Drive folder "VITASA_Enhanced"
src_drive = '/gdrive/My Drive/VITASA_Enhanced'

# Copy train_pair.py
!cp '{src_drive}/train_pair.py' VITASA_Enhanced/ 2>/dev/null && echo "✅ train_pair.py copied" || echo "⚠️  train_pair.py not found in Drive"

# Copy modules (nếu cần)
!cp -r '{src_drive}/text_normalization' VITASA_Enhanced/ 2>/dev/null && echo "✅ text_normalization copied" || echo "⚠️  text_normalization not found"
!cp -r '{src_drive}/imbalanced_learning' VITASA_Enhanced/ 2>/dev/null && echo "✅ imbalanced_learning copied" || echo "⚠️  imbalanced_learning not found"

# Verify
!echo "Files in VITASA_Enhanced:" && ls -la VITASA_Enhanced/ | grep -E '(train_pair|text_norm|imbalanced)'

In [ ]:
# 5. Smoke test: 1 epoch trên mobile
%cd VITASA_Enhanced

!python3 train_pair.py --domain mobile --loss ce --epochs 1 --subsample 0.1 --batch-size 64 2>&1 | tail -30

print("\n✅ Smoke test passed!")

In [ ]:
# 6. STRATEGY 1: Run BASELINE ONLY (C1) — safest, ~45 min for 3 domains
# Uncomment nếu muốn chỉ chạy baseline trước (recommended)

import subprocess
import time

EPOCHS = 10  # Safe for Colab Free
BATCH_SIZE = 64  # Larger batch = faster training

for domain in ['mobile', 'restaurant', 'hotel']:
    print(f"\n{'='*70}")
    print(f"[{domain}] Baseline (C1) — EPOCHS={EPOCHS}")
    print(f"{'='*70}")
    
    cmd = f"python3 train_pair.py --domain {domain} --loss ce --model visobert --epochs {EPOCHS} --batch-size {BATCH_SIZE}"
    result = subprocess.run(cmd.split(), capture_output=False)
    
    if result.returncode != 0:
        print(f"❌ ERROR: {domain} failed")
        break
    time.sleep(5)  # Wait before next domain

print("\n✅ Baseline runs complete!")

In [ ]:
# 7. STRATEGY 2 (resume-safe): Full ablation — 4 configs × 3 domains
# Backup lên Drive ngay sau mỗi config xong, tự skip config đã có kết quả.
# Dùng -u (unbuffered) để log/tqdm hiện ra NGAY LẬP TỨC trong Colab — nếu
# không có -u, Python buffer toàn bộ stdout và chỉ xả ra khi process thoát
# bình thường; nếu Colab bị ngắt giữa chừng (hết quota) thì MẤT SẠCH log,
# tưởng như không chạy gì dù thực ra đang train.

from google.colab import drive
drive.mount('/content/drive')

import os, shutil, subprocess, time
from pathlib import Path

DRIVE_BACKUP = Path('/content/drive/MyDrive/VITASA_pair_results_backup')
DRIVE_BACKUP.mkdir(parents=True, exist_ok=True)

LOCAL_RESULTS = Path('/content/VITASA_Enhanced/experiments/results_pair')
LOCAL_RESULTS.mkdir(parents=True, exist_ok=True)

MODEL = "visobert"
EPOCHS_BY_DOMAIN = {"mobile": 10, "restaurant": 5, "hotel": 5}
BATCH_SIZE = 64

CONFIGS = [
    ("C1_baseline", "--loss ce"),
    ("C2_norm",     "--loss ce --normalize"),
    ("C3_imbalanced","--loss focal"),
    ("C4_full",     "--loss focal --normalize"),
]

DOMAINS_TO_RUN = ["mobile", "restaurant", "hotel"]  # sửa lại nếu muốn chạy từng domain 1


def config_folder_name(domain, flags):
    """Phải khớp CHÍNH XÁC logic sinh config_name trong train_pair.py."""
    loss = "focal" if "focal" in flags else "ce"
    norm = "_norm" if "--normalize" in flags else ""
    return f"pair_{domain}_loss-{loss}{norm}_{MODEL}"


failed = []
for domain in DOMAINS_TO_RUN:
    epochs = EPOCHS_BY_DOMAIN.get(domain, 5)
    print(f"\n{'='*70}\nDomain: {domain} — epochs={epochs}\n{'='*70}")

    for config_name, flags in CONFIGS:
        folder = config_folder_name(domain, flags)
        drive_result = DRIVE_BACKUP / folder / "results.json"
        local_dir = LOCAL_RESULTS / folder
        local_result = local_dir / "results.json"

        if drive_result.exists():
            print(f"⏭  [{domain}/{config_name}] đã có kết quả trong Drive, bỏ qua. Copy về local...")
            local_dir.mkdir(parents=True, exist_ok=True)
            shutil.copy(drive_result, local_result)
            continue

        print(f"\n▶ [{domain}/{config_name}] bắt đầu — {epochs} epochs")
        start = time.time()

        cmd = (f"python3 -u train_pair.py --domain {domain} {flags} "
               f"--model {MODEL} --epochs {epochs} --batch-size {BATCH_SIZE}")

        env = {**os.environ, "PYTHONUNBUFFERED": "1"}
        proc = subprocess.Popen(
            cmd.split(), cwd="/content/VITASA_Enhanced", env=env,
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, bufsize=1,
        )
        for line in proc.stdout:
            print(line, end="")
        proc.wait()

        mins = (time.time() - start) / 60
        if proc.returncode != 0:
            failed.append(f"{domain}/{config_name}")
            print(f"❌ [{domain}/{config_name}] LỖI sau {mins:.1f} phút — không backup, tiếp tục config sau.")
            continue

        print(f"✅ [{domain}/{config_name}] xong sau {mins:.1f} phút — đang backup lên Drive...")

        drive_dir = DRIVE_BACKUP / folder
        drive_dir.mkdir(parents=True, exist_ok=True)
        if local_result.exists():
            shutil.copy(local_result, drive_result)
            print(f"   💾 Đã lưu → {drive_result}")
        else:
            print(f"   ⚠️  Không tìm thấy {local_result} để backup!")

        time.sleep(3)

print(f"\n{'='*70}")
print(f"Ablation complete. Failed: {len(failed)}")
if failed:
    print(f"  {failed}")
print("Nếu bị ngắt runtime giữa chừng: chỉ cần chạy lại TOÀN BỘ cell này,")
print("các config đã backup trong Drive sẽ tự động được skip.")
print(f"{'='*70}")


In [ ]:
# 8. Print summary table
import json
from pathlib import Path
from collections import defaultdict

results_dir = Path("experiments/results_pair")
results = defaultdict(dict)

BASELINE = {"mobile": 61.77, "restaurant": 41.12, "hotel": 52.64}

for results_file in sorted(results_dir.glob("*/results.json")):
    try:
        data = json.load(open(results_file))
        domain = data["domain"]
        config = data["config"]
        test_f1 = data["test"]["macro_f1"] * 100
        results[domain][config] = test_f1
    except Exception as e:
        print(f"[warn] {results_file}: {e}")

# Print table
print("\n" + "="*90)
print("ABLATION RESULTS — macro F1 (3 sentiment classes, loại 'none')")
print("="*90)

header = f"{'Domain':<12} {'C1_baseline':>15} {'C2_norm':>15} {'C3_focal':>15} {'C4_full':>15} {'Baseline':>13}"
print(header)
print("-" * 90)

for domain in ['mobile', 'restaurant', 'hotel']:
    baseline = BASELINE[domain]
    
    # Find results for each config
    c1_val = None
    c2_val = None
    c3_val = None
    c4_val = None
    
    for config_key, f1_val in results.get(domain, {}).items():
        if 'loss-ce_visobert' in config_key and 'norm' not in config_key:
            c1_val = f1_val
        elif 'loss-ce_norm' in config_key:
            c2_val = f1_val
        elif 'loss-focal_visobert' in config_key and 'norm' not in config_key:
            c3_val = f1_val
        elif 'loss-focal_norm' in config_key:
            c4_val = f1_val
    
    c1_str = f"{c1_val:.2f}%" if c1_val else "—"
    c2_str = f"{c2_val:.2f}%" if c2_val else "—"
    c3_str = f"{c3_val:.2f}%" if c3_val else "—"
    c4_str = f"{c4_val:.2f}%" if c4_val else "—"
    
    print(f"{domain:<12} {c1_str:>15} {c2_str:>15} {c3_str:>15} {c4_str:>15} {baseline:>12.2f}%")

print("="*90)
print(f"\n✅ Results saved to: experiments/results_pair/")
print(f"\n📥 Download: Run next cell to download all results as .tar.gz")

In [ ]:
# 9. Download results
from google.colab import files
import os

!tar -czf VITASA_pair_results.tar.gz experiments/results_pair/
!ls -lh VITASA_pair_results.tar.gz

print("\n📥 Downloading results...")
files.download("VITASA_pair_results.tar.gz")
print("✅ Done!")

## Notes

**Nếu timeout:**
1. Colab Free có thể timeout sau 12h hoặc disconnect nếu idle 30 phút
2. Nếu bị interrupt → restart từ cell tiếp theo (data vẫn ở)
3. Hoặc chạy 1 domain/lần thay vì tất cả cùng lúc

**Nếu muốn tăng accuracy:**
- Tăng EPOCHS từ 10 → 20 (nhưng có rủi ro timeout)
- Chạy trên Colab Pro (T4 unlimited hoặc V100)

**Cách lấy results:**
- Download .tar.gz → extract → xem `experiments/results_pair/*/results.json`
- Hoặc print summary từ cell trước khi download
